# Fraud Detection Model Training (v3 — ULB real anonymised transactions)

**Purpose.** Train the classifier behind `POST /v3/predict` and persist it as
`models/fraud_model_v3.joblib` using the same dict contract as v2
(`{"model", "features", "metrics", "dataset", ...}`), so the serving layer reads the
feature order from the artifact.

**Why a third model?** The v2 model scores near-perfectly on `card_transdata` because that
set is synthetic and its label is close to a deterministic function of two features. That
number is not evidence of fraud-detection capability. v3 trains on the **ULB Credit Card
Fraud Detection** set — 284,807 real European card transactions from September 2013 with
492 frauds (0.172 %) — which is the standard public benchmark for this problem and is
genuinely hard: the features are PCA components (V1–V28) plus `Amount`, so nothing can be
read off by eye, and the prevalence is two orders of magnitude lower.

**Dataset resolution.** The dataset is located on OpenML **by name** (`creditcard`,
version 1) with `sklearn.datasets.fetch_openml`, and the resolved id, row count and
class balance are asserted before training. The OpenML copy does **not** include the
`Time` column present in the Kaggle CSV; the 29 available features (V1–V28, Amount) are
used and this is recorded in the artifact metadata.

**Citation.** Andrea Dal Pozzolo, Olivier Caelen, Reid A. Johnson and Gianluca Bontempi.
*Calibrating Probability with Undersampling for Unbalanced Classification.* IEEE CIDM, 2015.
Dataset collected during a research collaboration between Worldline and the Machine Learning
Group of ULB (Université Libre de Bruxelles).

In [1]:
import json
import os
import platform
import time
from datetime import datetime, timezone

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.datasets import fetch_openml
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
OPENML_NAME = "creditcard"
OPENML_VERSION = 1
DATA_HOME = os.path.join("..", "data")
ARTIFACT_PATH = os.path.join("..", "models", "fraud_model_v3.joblib")
PRESETS_PATH = os.path.join("..", "app", "static", "data", "v3_presets.json")

EXPECTED_ROWS = 284_807
EXPECTED_FRAUD = 492
TARGET = "Class"

print("scikit-learn", sklearn.__version__, "| pandas", pd.__version__, "| python", platform.python_version())

scikit-learn 1.7.2 | pandas 2.3.3 | python 3.10.11


In [2]:
t0 = time.time()
bunch = fetch_openml(
    name=OPENML_NAME, version=OPENML_VERSION, as_frame=True, data_home=DATA_HOME, parser="auto"
)
download_seconds = round(time.time() - t0, 1)

df = bunch.frame.copy()
details = bunch.details or {}

# Verify that the name resolved to the dataset we expect before touching it.
assert df.shape[0] == EXPECTED_ROWS, f"unexpected row count {df.shape[0]}"
assert TARGET in df.columns, "target column 'Class' missing"
df[TARGET] = df[TARGET].astype(str).str.strip("'").astype(int)
assert int(df[TARGET].sum()) == EXPECTED_FRAUD, f"unexpected fraud count {int(df[TARGET].sum())}"

FEATURES = [c for c in df.columns if c != TARGET]
has_time = "Time" in FEATURES

dataset_info = {
    "openml_data_id": int(details.get("id")) if details.get("id") else None,
    "openml_name": details.get("name"),
    "openml_version": details.get("version"),
    "licence": details.get("licence"),
    "download_url": details.get("url"),
    "citation": "Dal Pozzolo, Caelen, Johnson, Bontempi. Calibrating Probability with Undersampling for Unbalanced Classification. IEEE CIDM 2015.",
    "n_rows_available": int(len(df)),
    "n_features": len(FEATURES),
    "features_note": "OpenML copy omits the Kaggle 'Time' column; V1-V28 + Amount used." if not has_time else "Time + V1-V28 + Amount",
    "fraud_count": int(df[TARGET].sum()),
    "fraud_prevalence": round(float(df[TARGET].mean()), 5),
    "download_seconds": download_seconds,
    "description": "Real, anonymised European card transactions (Sept 2013); PCA-transformed features.",
}
print(json.dumps(dataset_info, indent=2))
print()
print(f"{len(FEATURES)} features: {FEATURES[:5]} ... {FEATURES[-2:]}")
print("Class balance:")
print(df[TARGET].value_counts().rename("count"))
df[["V1", "V2", "V14", "Amount", TARGET]].describe().T.round(3)

{
  "openml_data_id": 1597,
  "openml_name": "creditcard",
  "openml_version": "1",
  "licence": "Public",
  "download_url": "https://openml.org/data/v1/download/1673544/creditcard.arff",
  "citation": "Dal Pozzolo, Caelen, Johnson, Bontempi. Calibrating Probability with Undersampling for Unbalanced Classification. IEEE CIDM 2015.",
  "n_rows_available": 284807,
  "n_features": 29,
  "features_note": "OpenML copy omits the Kaggle 'Time' column; V1-V28 + Amount used.",
  "fraud_count": 492,
  "fraud_prevalence": 0.00173,
  "download_seconds": 4.1,
  "description": "Real, anonymised European card transactions (Sept 2013); PCA-transformed features."
}

29 features: ['V1', 'V2', 'V3', 'V4', 'V5'] ... ['V28', 'Amount']
Class balance:
Class
0    284315
1       492
Name: count, dtype: int64


,count,mean,std,min,25%,50%,75%,max
V1,284807.0,0.000,1.959,-56.408,-0.920,0.018,1.316,2.455
V2,284807.0,0.000,1.651,-72.716,-0.599,0.065,0.804,22.058
V14,284807.0,0.000,0.959,-19.214,-0.426,0.051,0.493,10.527
Amount,284807.0,88.350,250.120,0.000,5.600,22.000,77.165,25691.160
Class,284807.0,0.002,0.042,0.000,0.000,0.000,0.000,1.000


## Class imbalance: 0.172 % positives

At this prevalence a constant "legitimate" predictor is 99.83 % accurate and catches
nothing. Accuracy is therefore not reported as a headline anywhere in this project.

* `class_weight="balanced"` re-weights each fraud row by roughly 580× so the forest
  cannot ignore the minority class.
* The headline metrics are **fraud-class precision, recall, F1** at the default 0.5
  threshold and **PR-AUC (average precision)** across all thresholds. PR-AUC is computed
  only from how well positives are retrieved and how clean the retrieved set is, and unlike
  ROC-AUC it is not inflated by the ~57,000 easy true negatives in the hold-out set.
* Published results on this benchmark land in the PR-AUC ≈ 0.75–0.87 range for tree
  ensembles without heavy feature work. Nothing below is tuned: default `n_estimators`,
  no depth limit, no threshold search, no resampling. The number is reported as it comes out.

In [3]:
X = df[FEATURES].to_numpy(dtype=float)
y = df[TARGET].to_numpy(dtype=int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(f"train rows: {len(X_train):,} (fraud {int(y_train.sum())}, {y_train.mean():.4%})")
print(f"test  rows: {len(X_test):,} (fraud {int(y_test.sum())}, {y_test.mean():.4%})")

train rows: 227,845 (fraud 394, 0.1729%)
test  rows: 56,962 (fraud 98, 0.1720%)


In [4]:
model = RandomForestClassifier(class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE)

t0 = time.time()
model.fit(X_train, y_train)
fit_seconds = round(time.time() - t0, 1)

n_nodes = int(sum(est.tree_.node_count for est in model.estimators_))
max_depth = int(max(est.tree_.max_depth for est in model.estimators_))
print(f"fit time: {fit_seconds}s | trees: {model.n_estimators} | total nodes: {n_nodes:,} | deepest tree: {max_depth}")

fit time: 40.7s | trees: 100 | total nodes: 44,358 | deepest tree: 43


In [5]:
y_proba = model.predict_proba(X_test)[:, 1]
y_pred = (y_proba >= 0.5).astype(int)

print(classification_report(y_test, y_pred, target_names=["legit", "fraud"], digits=4))

precision, recall, f1, _ = precision_recall_fscore_support(
    y_test, y_pred, average="binary", pos_label=1
)
pr_auc = average_precision_score(y_test, y_proba)
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
tn, fp, fn, tp = (int(v) for v in cm.ravel())

print(f"fraud precision : {precision:.4f}")
print(f"fraud recall    : {recall:.4f}")
print(f"fraud F1        : {f1:.4f}")
print(f"PR-AUC (AP)     : {pr_auc:.4f}")
print()
print("Confusion matrix (rows = actual, cols = predicted):")
print(pd.DataFrame(cm, index=["actual legit", "actual fraud"], columns=["pred legit", "pred fraud"]))
print()

importances = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)
print("Top-10 feature importances (Gini):")
print(importances.head(10).round(4).to_string())

# Recall at a few alternative thresholds, to show what the /threshold field trades away.
print()
print("threshold  precision  recall")
for thr in (0.1, 0.3, 0.5, 0.7, 0.9):
    p_thr, r_thr, _, _ = precision_recall_fscore_support(
        y_test, (y_proba >= thr).astype(int), average="binary", pos_label=1, zero_division=0
    )
    print(f"   {thr:.1f}      {p_thr:.4f}   {r_thr:.4f}")

metrics = {
    "fraud_precision": round(float(precision), 4),
    "fraud_recall": round(float(recall), 4),
    "fraud_f1": round(float(f1), 4),
    "pr_auc": round(float(pr_auc), 4),
    "threshold": 0.5,
    "confusion_matrix": {"tn": tn, "fp": fp, "fn": fn, "tp": tp},
    "feature_importances": {k: round(float(v), 4) for k, v in importances.items()},
    "n_train_rows": int(len(X_train)),
    "n_test_rows": int(len(X_test)),
    "n_test_fraud": int(y_test.sum()),
}
print()
print("METRICS_JSON", json.dumps({k: v for k, v in metrics.items() if k != "feature_importances"}))

              precision    recall  f1-score   support

       legit     0.9996    0.9999    0.9998     56864
       fraud     0.9487    0.7551    0.8409        98

    accuracy                         0.9995     56962
   macro avg     0.9741    0.8775    0.9203     56962
weighted avg     0.9995    0.9995    0.9995     56962



fraud precision : 0.9487
fraud recall    : 0.7551
fraud F1        : 0.8409
PR-AUC (AP)     : 0.8485

Confusion matrix (rows = actual, cols = predicted):
              pred legit  pred fraud
actual legit       56860           4
actual fraud          24          74

Top-10 feature importances (Gini):
V4     0.1484
V10    0.1478
V14    0.1390
V12    0.0915
V11    0.0875
V17    0.0606
V7     0.0465
V3     0.0458
V16    0.0375
V2     0.0173

threshold  precision  recall
   0.1      0.7611   0.8776
   0.3      0.9302   0.8163
   0.5      0.9487   0.7551
   0.7      0.9718   0.7041
   0.9      0.9783   0.4592

METRICS_JSON {"fraud_precision": 0.9487, "fraud_recall": 0.7551, "fraud_f1": 0.8409, "pr_auc": 0.8485, "threshold": 0.5, "confusion_matrix": {"tn": 56860, "fp": 4, "fn": 24, "tp": 74}, "n_train_rows": 227845, "n_test_rows": 56962, "n_test_fraud": 98}


## Preset transactions for the demo UI

Twenty-nine PCA components are not something a visitor can type in, so the landing page
drives v3 through three **real hold-out rows** embedded as JSON at build time:

1. a routine legitimate transaction the model is confident about,
2. a legitimate transaction the model finds *unusual* (probability between 0.15 and 0.45 —
   moving the threshold slider flips its verdict), and
3. a fraudulent transaction the model catches at 0.5 but would miss at a strict threshold.

Ground-truth labels and the model's hold-out probability are stored alongside the values so
the UI can show both honestly.

In [6]:
def pick(mask, prefer_index=0):
    idx = np.flatnonzero(mask)
    if len(idx) == 0:
        raise RuntimeError("no hold-out row matches the preset criteria")
    # Deterministic choice: sort by probability so re-runs pick the same row.
    order = idx[np.argsort(y_proba[idx], kind="stable")]
    return int(order[min(prefer_index, len(order) - 1)])


legit_routine_idx = pick((y_test == 0) & (y_proba < 0.02) & (X_test[:, FEATURES.index("Amount")] > 5), prefer_index=100)
legit_unusual_idx = pick((y_test == 0) & (y_proba > 0.15) & (y_proba < 0.45), prefer_index=len(np.flatnonzero((y_test == 0) & (y_proba > 0.15) & (y_proba < 0.45))) // 2)
fraud_caught_idx = pick((y_test == 1) & (y_proba > 0.55) & (y_proba < 0.9), prefer_index=len(np.flatnonzero((y_test == 1) & (y_proba > 0.55) & (y_proba < 0.9))) // 2)

presets = []
for key, idx, title, blurb in (
    ("legit_routine", legit_routine_idx, "Legitimate — routine",
     "A typical purchase. The model assigns it a very low fraud probability."),
    ("legit_unusual", legit_unusual_idx, "Legitimate — unusual",
     "Genuine, but its PCA profile partly resembles fraud. Lower the threshold and it gets flagged: this is the false-positive cost of higher recall."),
    ("fraud_caught", fraud_caught_idx, "Fraudulent — caught at 0.5",
     "A confirmed fraud the model catches at the default threshold. Raise the threshold past its probability and it slips through: the cost of higher precision."),
):
    row = X_test[idx]
    presets.append({
        "id": key,
        "title": title,
        "description": blurb,
        "ground_truth": int(y_test[idx]),
        "holdout_probability": round(float(y_proba[idx]), 4),
        "amount": round(float(row[FEATURES.index("Amount")]), 2),
        "values": {name: round(float(v), 6) for name, v in zip(FEATURES, row)},
    })
    print(f"{key:14s} truth={int(y_test[idx])} proba={y_proba[idx]:.4f} amount={row[FEATURES.index('Amount')]:.2f}")

os.makedirs(os.path.dirname(PRESETS_PATH), exist_ok=True)
with open(PRESETS_PATH, "w", encoding="utf-8") as fh:
    json.dump({"features": FEATURES, "model_version": "joblib_model_v3", "presets": presets}, fh, indent=2)
print("wrote", PRESETS_PATH)

legit_routine  truth=0 proba=0.0000 amount=12.99
legit_unusual  truth=0 proba=0.2100 amount=0.77
fraud_caught   truth=1 proba=0.8200 amount=1.00
wrote ..\app\static\data\v3_presets.json


In [7]:
os.makedirs(os.path.dirname(ARTIFACT_PATH), exist_ok=True)

artifact = {
    "model": model,
    "features": FEATURES,
    "target": TARGET,
    "metrics": metrics,
    "dataset": dataset_info,
    "subsampled": False,
    "n_rows_used": int(len(df)),
    "sklearn_version": sklearn.__version__,
    "trained_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "fit_seconds": fit_seconds,
    "model_params": {"class_weight": "balanced", "n_estimators": model.n_estimators, "random_state": RANDOM_STATE},
}
joblib.dump(artifact, ARTIFACT_PATH, compress=3)
size_mb = os.path.getsize(ARTIFACT_PATH) / 1e6
print(f"Saved {ARTIFACT_PATH} ({size_mb:.2f} MB, compress=3)")

reloaded = joblib.load(ARTIFACT_PATH)
assert reloaded["features"] == FEATURES, "feature order drifted between save and load"
for preset in presets:
    vec = np.array([[preset["values"][name] for name in FEATURES]])
    proba = float(reloaded["model"].predict_proba(vec)[0, 1])
    print(f"round-trip {preset['id']:14s} proba={proba:.4f} (stored {preset['holdout_probability']:.4f})")

Saved ..\models\fraud_model_v3.joblib (1.12 MB, compress=3)


round-trip legit_routine  proba=0.0000 (stored 0.0000)
round-trip legit_unusual  proba=0.2100 (stored 0.2100)
round-trip fraud_caught   proba=0.8200 (stored 0.8200)
